# Expliquer un détecteur d'intrusion (UNSW-NB15)

Projet du module Machine Learning pour la cyber-sécurité, cahier des charges n° 1.

Binôme : *à compléter*

Ce notebook est à la fois le code et le rapport. Il s'exécute de haut en bas, dans l'ordre, sans intervention manuelle.

### Résumé

Nous entraînons trois détecteurs d'intrusion (régression logistique, forêt aléatoire, XGBoost) sur les flux réseau d'UNSW-NB15, puis nous cherchons à comprendre sur quoi ils fondent leurs alertes.

- Après nettoyage (suppression d'une colonne qui révèle l'étiquette, des doublons, des contradictions et du recouvrement entre fichiers), les modèles d'arbres atteignent une PR-AUC de 0,985 en validation et d'environ 0,945 sur le test.
- Sur le test, leur précision chute (0,93 puis 0,69 pour XGBoost) : près d'un flux normal sur quatre est signalé. Les fausses alertes se concentrent presque toutes sur deux valeurs du TTL source.
- Les trois mesures d'importance (interne, permutation, SHAP) placent les variables TTL en tête ; mélangées ensemble, elles portent la moitié de l'avance de la forêt sur le hasard. Les cinq alertes expliquées reposent toutes d'abord sur `sttl`.
- Retirer les variables TTL ne change presque rien aux scores ni à la forme des erreurs : l'identité des machines du banc d'essai est reconstruite parfaitement (ROC-AUC de 1,000) à partir des autres variables.
- Ces résultats sont stables sur neuf entraînements avec d'autres graines et d'autres découpages.

**Conclusion :** sur UNSW-NB15, les modèles apprennent en grande partie à reconnaître quelles machines du banc d'essai ont produit le trafic, un artefact de collecte, plutôt que des comportements d'attaque transposables à un autre réseau.

## 0. Mise en place

In [ ]:
import sys
from pathlib import Path

# Le notebook est dans notebooks/, le code partagé dans src/ à la racine du projet.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import time

import numpy as np
import pandas as pd
import sklearn, xgboost, shap

from src import config
import matplotlib.pyplot as plt

from src import data as D
from src.preprocessing import build_preprocessor, numeric_columns
from src import models as M
from src import evaluation as E
from src import explain as X
from src import stability as S

config.set_seed()
print("Python", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| shap", shap.__version__)
print("Graine :", config.SEED)

## 1. Problématique

Un centre opérationnel de sécurité (SOC) reçoit les alertes d'un détecteur d'intrusion. Pour agir, l'analyste doit comprendre pourquoi une connexion a été jugée malveillante : isoler une machine, alerter un utilisateur ou classer l'alerte sans suite sont des décisions qui demandent une justification. Un détecteur qui lève des alertes sans raison lisible est difficile à exploiter.

Il y a un second risque, moins visible. Un modèle d'apprentissage automatique prend le chemin le plus facile vers un bon score. Si, dans les données d'entraînement, les attaques se distinguent du trafic normal par un détail lié à la façon dont les données ont été collectées (la configuration des machines du banc d'essai, l'outil de capture, l'ordre d'enregistrement), le modèle utilisera ce détail. Il paraîtra excellent sur ce jeu de données et échouera sur un vrai réseau. Les jeux publics de détection d'intrusion affichent souvent des scores supérieurs à 99 % ; la question est de savoir ce que ces scores mesurent vraiment.

**Question centrale.** Un détecteur entraîné sur UNSW-NB15 apprend-il des comportements d'attaque, ou des artefacts de collecte ?

**Démarche.**

1. Préparer les données sans fuite d'information et entraîner trois modèles de complexité croissante (sections 2 et 3).
2. Les évaluer avec des métriques adaptées, en choisissant le seuil sur la validation et en n'utilisant le test qu'une fois (section 4).
3. Expliquer leurs décisions : trois mesures d'importance globales, cinq alertes expliquées individuellement avec SHAP (sections 5.1 et 5.2).
4. Juger la plausibilité des variables importantes du point de vue réseau, puis ré-entraîner sans les variables suspectes (sections 5.3 et 5.4).
5. Vérifier que les explications ne dépendent ni de la graine ni du découpage (section 5.5).

La section 6 répond à la question centrale à partir de ces éléments.

## 2. Données

### 2.1 Description du jeu

Nous utilisons UNSW-NB15 (Moustafa et Slay, 2015), produit par le Cyber Range Lab d'UNSW Canberra. Le trafic a été généré sur un banc d'essai mêlant trafic normal réaliste et attaques synthétiques, puis découpé en flux réseau. Chaque flux est décrit par 42 variables : durée, octets et paquets dans chaque sens, TTL, fenêtres TCP, délais, ainsi que des compteurs de connexions récentes (variables `ct_*`). Trois variables sont textuelles : le protocole (`proto`), le service applicatif (`service`) et l'état de la connexion (`state`).

Les auteurs fournissent deux fichiers déjà séparés, un fichier d'entraînement et un fichier de test. Chaque ligne porte une étiquette binaire (`label`, 1 = attaque) et la famille d'attaque (`attack_cat`). Notre modèle principal est binaire. La famille ne sert qu'à l'analyse, jamais comme variable d'entrée.

In [ ]:
raw_train, raw_test = D.load_raw()

for nom, df in [("entraînement", raw_train), ("test", raw_test)]:
    print(f"Fichier {nom} : {len(df):,} lignes, {df.shape[1]} colonnes, "
          f"{df[config.TARGET].mean():.1%} d'attaques")

print("\nTypes de colonnes :", raw_train.dtypes.value_counts().to_dict())

In [ ]:
familles = pd.DataFrame({
    "entraînement": raw_train[config.ATTACK_CAT].value_counts(),
    "test": raw_test[config.ATTACK_CAT].value_counts(),
})
familles

Deux remarques dès cette étape.

Les attaques sont **majoritaires** dans les deux fichiers, alors que dans un vrai réseau elles sont rares. Les auteurs n'ont conservé qu'un échantillon du trafic normal. Conséquence pour l'évaluation : la PR-AUC d'un classifieur aléatoire est égale à la proportion d'attaques, il faudra donc toujours lire nos PR-AUC par rapport à ce niveau de référence.

Les familles sont très déséquilibrées entre elles (Generic : 40 000 flux d'entraînement, Worms : 130). Les familles rares seront à surveiller dans l'analyse des erreurs.

### 2.2 Nettoyage

Les fichiers ne contiennent ni valeur manquante, ni valeur infinie, ni valeur négative :

In [ ]:
for nom, df in [("entraînement", raw_train), ("test", raw_test)]:
    num = df.select_dtypes("number")
    print(f"{nom} : manquantes = {int(df.isna().sum().sum())}, "
          f"infinies = {int(np.isinf(num).sum().sum())}, "
          f"négatives = {int((num < 0).sum().sum())}")

En revanche, quatre problèmes demandent une décision.

**La colonne `id` révèle l'étiquette.** Les lignes sont rangées par `id`, et cet ordre suit l'étiquette : la proportion d'attaques par quart du fichier d'entraînement passe de 0 % à 100 %. Un modèle qui verrait `id` apprendrait l'ordre du fichier et non le trafic. Nous la supprimons.

In [ ]:
quarts = pd.qcut(raw_train["id"], 4, labels=["1er quart", "2e quart", "3e quart", "4e quart"])
raw_train.groupby(quarts, observed=True)[config.TARGET].mean().round(3).rename("part d'attaques")

**Doublons.** Beaucoup de lignes sont des copies exactes d'une autre (mêmes variables, même étiquette). Les garder fausserait la validation croisée, car des copies d'un même flux se retrouveraient des deux côtés d'un pli. Nous gardons une seule ligne par copie. Nous supprimons les doublons **avant** tout découpage, sinon ils pourraient fuir entre entraînement et validation.

**Contradictions.** Certains vecteurs de variables apparaissent à la fois avec l'étiquette « normal » et l'étiquette « attaque ». Aucun modèle ne peut classer correctement les deux. Ces lignes illustrent le chevauchement de classes décrit par Zoghi et Serpen (2021) dans leur analyse du jeu. Nous les retirons toutes.

La même fonction est appliquée aux deux fichiers.

In [ ]:
train_clean, rapport_train = D.clean(raw_train)
test_clean, rapport_test = D.clean(raw_test)
pd.DataFrame({"entraînement": rapport_train, "test": rapport_test})

**Recouvrement entre les deux fichiers.** Des flux du fichier de test ont exactement les mêmes variables qu'un flux du fichier d'entraînement. Un modèle les classerait juste par simple mémoire, ce qui mesurerait la mémoire et non la généralisation. Nous les retirons du test. La comparaison se fait sur les variables seulement, quelle que soit l'étiquette.

In [ ]:
test_final, rapport_recouvrement = D.remove_overlap(test_clean, raw_train)
rapport_recouvrement

**Valeurs textuelles inconnues.** La variable `state` prend dans le test deux valeurs absentes de l'entraînement (`ACC` et `CLO`). L'encodage one-hot devra les accepter sans erreur : une valeur inconnue sera codée par des zéros. Par ailleurs, `service` vaut `-` pour environ la moitié des flux. Ce n'est pas une valeur manquante mais une vraie catégorie, « aucun service reconnu », que nous conservons telle quelle.

In [ ]:
for col in config.CATEGORICAL:
    inconnues = sorted(set(test_final[col]) - set(train_clean[col]))
    print(f"{col} : {train_clean[col].nunique()} valeurs en entraînement, inconnues dans le test : {inconnues}")

### 2.3 Découpage entraînement / validation / test

**Pas de découpage chronologique possible.** Le cahier des charges demande de respecter l'ordre du temps quand les données sont horodatées. Les fichiers de partition d'UNSW-NB15 ne contiennent pas les colonnes de temps (`stime`, `ltime`) du jeu complet. Nous conservons donc la séparation officielle des auteurs : le fichier de test nettoyé sert de jeu de test.

**Validation.** Le fichier d'entraînement nettoyé est coupé en 80 % pour l'entraînement et 20 % pour la validation. Le découpage est stratifié sur la famille d'attaque, pour que les familles rares soient présentes des deux côtés dans les mêmes proportions. Comme « Normal » est une des familles, la proportion d'attaques est conservée aussi.

Rôle de chaque jeu :
- l'**entraînement** sert à ajuster les modèles et, par validation croisée, à régler leurs hyperparamètres ;
- la **validation** sert à comparer les modèles réglés et à choisir un seuil de décision ;
- le **test** n'est utilisé qu'une seule fois, pour les résultats finaux.

In [ ]:
train, val = D.split_train_val(train_clean)
test = test_final

resume = pd.DataFrame({
    nom: {"flux": len(df), "part d'attaques": round(df[config.TARGET].mean(), 3),
          "Worms": int((df[config.ATTACK_CAT] == "Worms").sum())}
    for nom, df in [("entraînement", train), ("validation", val), ("test", test)]
}).T
resume = resume.astype({"flux": int, "Worms": int})
resume

In [ ]:
# Vérifications : aucun vecteur de variables partagé entre les trois jeux.
cols = D.feature_columns(train)
def communs(a, b):
    return len(a[cols].drop_duplicates().merge(b[cols].drop_duplicates()))
print("entraînement et validation :", communs(train, val))
print("entraînement et test       :", communs(train, test))
print("validation et test         :", communs(val, test))

In [ ]:
part = pd.DataFrame({nom: df[config.ATTACK_CAT].value_counts(normalize=True)
                     for nom, df in [("entraînement", train), ("validation", val), ("test", test)]})
ax = part.sort_values("entraînement").plot.barh(figsize=(8, 5))
ax.set_xlabel("part des flux")
ax.set_title("Répartition des familles après nettoyage")
plt.tight_layout()
plt.savefig(config.FIGURES / "familles_apres_nettoyage.png", dpi=150)
plt.show()

Après nettoyage, la part d'attaques baisse nettement dans les deux fichiers : beaucoup de doublons étaient des attaques, en particulier de la famille Generic. Le test garde une répartition différente de l'entraînement : beaucoup plus de trafic normal (64 % contre 51 %), moins d'Exploits et de Fuzzers, un peu plus de Generic. C'est un écart réel entre les deux fichiers officiels, dont il faudra tenir compte en lisant les résultats sur le test.

**Prétraitement.** L'encodage one-hot des variables textuelles et, pour la régression logistique seulement, la standardisation des variables numériques sont placés dans un `Pipeline` scikit-learn avec chaque modèle (voir `src/preprocessing.py`). Ils sont donc ajustés sur l'entraînement uniquement, et à l'intérieur de chaque pli pendant la validation croisée. Aucune opération apprise ne voit la validation ni le test.

In [ ]:
X_train, y_train, fam_train = D.get_xy(train)
X_val, y_val, fam_val = D.get_xy(val)
X_test, y_test, fam_test = D.get_xy(test)

# Contrôle : le prétraitement s'ajuste sur l'entraînement et s'applique au test sans erreur.
prep = build_preprocessor(numeric_columns(X_train), scale=True).fit(X_train)
print("Variables après encodage :", prep.transform(X_train).shape[1])
print("Forme du test encodé     :", prep.transform(X_test).shape)

## 3. Modèles

### Protocole commun

Les trois modèles suivent le même protocole, pour que la comparaison soit équitable.

- Chaque modèle est un `Pipeline` : prétraitement (section 2.3) puis classifieur. Le prétraitement est donc réajusté dans chaque pli de validation croisée.
- Les hyperparamètres sont réglés par **validation croisée stratifiée à 5 plis sur le jeu d'entraînement seulement**. La validation et le test ne servent pas au réglage.
- Le critère de réglage est la **PR-AUC** (`average_precision`). Elle juge le classement des flux par score sur tous les seuils, sans dépendre du seuil de 0,5, et elle reste adaptée au déséquilibre des classes.
- La graine est fixée à 42 partout (découpage des plis, tirage des combinaisons, aléa interne des modèles).

Pour la régression logistique, les cinq valeurs possibles sont toutes essayées (recherche en grille). Pour la forêt et XGBoost, les combinaisons possibles se comptent par milliers. Nous en tirons 12 au hasard (recherche aléatoire), soit 60 entraînements par modèle avec les 5 plis. C'est un compromis assumé avec le temps de calcul sur processeur.

### Mise en cache des recherches

La recherche complète prend environ une demi-heure sur processeur, dont plus de vingt minutes pour la forêt. Pour ne pas la refaire à chaque exécution du notebook, son résultat est enregistré dans le dossier `models/` et rechargé ensuite (voir `src/cache.py`).

Un résultat enregistré n'est réutilisé que s'il a été produit à partir des mêmes entrées. Chaque fichier porte une empreinte calculée sur les données d'entraînement, l'espace de recherche, la graine, le nombre de plis et les versions des bibliothèques. Si l'un de ces éléments change, l'empreinte ne correspond plus et la recherche est relancée. Le paramètre `config.RETUNE = True` force un nouveau calcul complet ; c'est ainsi que sera faite l'exécution finale avant le rendu. Le dossier `models/` n'est pas versionné : sur une copie neuve du dépôt, la première exécution refait tout.

In [ ]:
pipelines = M.make_pipelines(numeric_columns(X_train))
recherches = {}
duree_recherche = {}

def regler(nom):
    recherche, secondes, depuis_cache = M.tune_cached(nom, pipelines[nom], X_train, y_train)
    recherches[nom], duree_recherche[nom] = recherche, secondes
    moyenne, ecart = M.cv_summary(recherche)
    origine = "rechargée depuis le cache" if depuis_cache else "calculée"
    print(f"{nom} : PR-AUC en validation croisée = {moyenne:.4f} ± {ecart:.4f}")
    print(f"Recherche {origine} (durée de la recherche : {secondes / 60:.1f} min)")
    print("Meilleurs réglages :", M.short_params(recherche.best_params_))

### 3.1 Baseline : régression logistique

La régression logistique attribue un poids à chaque variable, additionne les contributions et transforme la somme en probabilité. Elle ne peut tracer qu'une frontière linéaire entre les classes. C'est ce qui en fait une bonne référence : les modèles plus complexes doivent la battre nettement pour justifier leur coût. Les variables numériques sont standardisées pour elle. Le seul hyperparamètre réglé est `C`, l'inverse de la force de régularisation.

In [ ]:
regler("Régression logistique")

### 3.2 Forêt aléatoire

Une forêt aléatoire combine des centaines d'arbres de décision. Chaque arbre est entraîné sur un échantillon tiré au hasard des lignes et ne considère qu'une partie des variables à chaque division. Le vote moyen des arbres corrige une grande partie des erreurs individuelles. Réglages explorés : nombre d'arbres, profondeur maximale, taille minimale des feuilles et part des variables considérées à chaque division.

In [ ]:
regler("Forêt aléatoire")

### 3.3 XGBoost

XGBoost construit aussi des arbres, mais l'un après l'autre : chaque nouvel arbre corrige les erreurs des précédents (gradient boosting). Réglages explorés : nombre d'arbres, profondeur, taux d'apprentissage, part des lignes et des variables tirées pour chaque arbre, et poids minimal d'une feuille.

In [ ]:
regler("XGBoost")

### 3.4 Comparaison

Le tableau rassemble, pour chaque modèle réglé :

- la PR-AUC moyenne en validation croisée et son écart-type entre plis ;
- la PR-AUC et le F1 (au seuil par défaut de 0,5) sur le **jeu de validation**, que les modèles n'ont jamais vu ;
- le temps d'entraînement sur tout le jeu d'entraînement, sur un seul cœur pour que les trois modèles soient mesurés dans les mêmes conditions ;
- le temps d'inférence pour 1 000 flux.

Les temps sont mis en cache avec les recherches : ceux affichés sont ceux de l'exécution qui a fait les mesures.

Le niveau de référence de la PR-AUC est la proportion d'attaques : environ 0,49 en validation. Le test n'est toujours pas utilisé ; l'évaluation complète fait l'objet de la section 4.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score

lignes = {}
for nom, recherche in recherches.items():
    modele = recherche.best_estimator_
    score_val = modele.predict_proba(X_val)[:, 1]
    cv_moy, cv_ecart = M.cv_summary(recherche)
    t_fit, t_pred, _ = M.time_model_cached(nom, modele, X_train, y_train, X_val)
    lignes[nom] = {
        "PR-AUC CV (moyenne)": round(cv_moy, 4),
        "PR-AUC CV (écart-type)": round(cv_ecart, 4),
        "PR-AUC validation": round(average_precision_score(y_val, score_val), 4),
        "F1 validation (seuil 0,5)": round(f1_score(y_val, score_val >= 0.5), 4),
        "entraînement (s)": round(t_fit, 1),
        "inférence (ms / 1 000 flux)": round(t_pred, 2),
    }

comparaison = pd.DataFrame(lignes).T
print(f"Niveau de référence de la PR-AUC en validation : {y_val.mean():.3f}")
comparaison

In [ ]:
reglages = {nom: {k: str(v) for k, v in M.short_params(r.best_params_).items()}
            for nom, r in recherches.items()}
pd.DataFrame(reglages).T.fillna("")

**Lecture des résultats.**

Les trois modèles sont très loin au-dessus du niveau de référence (PR-AUC de 0,488 pour un classement au hasard en validation). Les scores de validation croisée et ceux du jeu de validation sont presque identiques, avec des écarts-types entre plis très faibles. Les modèles ne sur-apprennent donc pas leurs plis d'entraînement, et le réglage est stable.

**Les arbres battent nettement la baseline.** La régression logistique atteint une PR-AUC de 0,959 en validation, la forêt et XGBoost 0,985. Exprimé en erreur restante (1 − PR-AUC), les modèles d'arbres la réduisent d'environ deux tiers, de 0,041 à 0,015. Le F1 au seuil de 0,5 passe de 0,904 à 0,933. Les relations entre variables et attaque ne sont donc pas linéaires, ce que les arbres savent capter. La meilleure valeur de `C` pour la régression logistique est à la limite de la grille (100). Un essai séparé avec `C` de 10 à 10 000 donne la même PR-AUC à 0,0002 près : la régularisation ne joue presque pas, c'est la forme linéaire du modèle qui le limite.

**La forêt et XGBoost ne se départagent pas en performance.** L'écart de PR-AUC en validation (0,0002) est plus petit que l'écart-type entre plis (environ 0,001). Conclure que l'un est meilleur serait lire du bruit. La meilleure profondeur de XGBoost (10) est aussi la plus grande valeur explorée ; l'étendre pourrait gagner quelques millièmes, sans changer la conclusion vu la proximité des deux modèles.

**Ils se départagent en coût d'entraînement, pas d'inférence.** La forêt est de loin la plus lente à entraîner : environ 120 secondes sur un cœur, contre 14 pour XGBoost et 5 pour la régression logistique dans l'exécution enregistrée. Sur deux exécutions complètes, ces durées ont varié presque du simple au double selon la charge de la machine, mais l'ordre est resté le même, avec une forêt huit à quatorze fois plus lente que XGBoost. En inférence, la forêt et XGBoost sont du même ordre, entre 10 et 35 ms pour 1 000 flux selon l'exécution, et leur classement s'est inversé d'une exécution à l'autre : nous ne concluons donc pas sur ce point. Les trois modèles restent largement compatibles avec une détection quasi temps réel, à moins de 0,05 ms par flux.

**Prudence.** Des scores aussi élevés sont précisément ce que ce sujet nous demande de questionner. La section 2 a déjà montré qu'une seule variable, `sttl`, sépare presque parfaitement certaines valeurs de trafic normal et d'attaque. Avant de croire que ces modèles reconnaissent des comportements d'attaque, il faudra vérifier sur quoi ils s'appuient (section 5).

In [ ]:
# Les modèles réglés, réutilisés dans les sections suivantes.
modeles = {nom: r.best_estimator_ for nom, r in recherches.items()}

## 4. Évaluation

### Protocole

L'évaluation se fait en deux temps.

1. **Sur la validation**, nous choisissons pour chaque modèle un seuil de décision : celui qui maximise le F1. Le seuil de 0,5 n'a rien de particulier ; un score n'est pas une probabilité calibrée, et chaque modèle a son propre seuil utile. La règle de choix a été fixée avant d'ouvrir le test, et n'a pas été modifiée ensuite.
2. **Sur le test**, que nous ouvrons ici une seule fois, nous appliquons ces seuils sans rien changer, et nous rapportons toutes les métriques demandées : matrices de confusion, précision, rappel et F1 par classe, courbes précision-rappel et PR-AUC, ROC-AUC en complément.

Les analyses qui suivent l'ouverture du test servent à comprendre les résultats. Aucune ne sert à modifier un modèle ou un seuil.

### 4.1 Choix du seuil sur la validation

In [ ]:
scores_val = {nom: m.predict_proba(X_val)[:, 1] for nom, m in modeles.items()}
seuils = {nom: E.best_f1_threshold(y_val, s)[0] for nom, s in scores_val.items()}

resultats_val = pd.DataFrame({nom: E.summary(y_val, scores_val[nom], seuils[nom]) for nom in modeles}).T
resultats_val.insert(0, "seuil", pd.Series(seuils))
resultats_val.round(4)

In [ ]:
E.plot_pr_curves(y_val, scores_val, "Courbes précision-rappel, validation",
                 "pr_validation.png", thresholds=seuils)

Les points marquent le seuil retenu pour chaque modèle.

### 4.2 Résultats sur le test

Le jeu de test est utilisé ici pour la première fois.

In [ ]:
scores_test = {nom: m.predict_proba(X_test)[:, 1] for nom, m in modeles.items()}

resultats_test = pd.DataFrame({nom: E.summary(y_test, scores_test[nom], seuils[nom]) for nom in modeles}).T
resultats_test.insert(0, "seuil", pd.Series(seuils))
print(f"Part d'attaques dans le test (PR-AUC d'un modèle au hasard) : {y_test.mean():.3f}")
resultats_test.round(4)

#### Matrices de confusion

In [ ]:
E.plot_confusion_matrices(y_test, scores_test, seuils, "matrices_confusion_test.png")

#### Précision, rappel et F1 par classe

In [ ]:
pd.concat({nom: E.per_class_report(y_test, scores_test[nom], seuils[nom]) for nom in modeles}).round(4)

#### Courbes précision-rappel et PR-AUC

In [ ]:
E.plot_pr_curves(y_test, scores_test, "Courbes précision-rappel, test",
                 "pr_test.png", thresholds=seuils)

#### Validation contre test

In [ ]:
ecart = pd.DataFrame({
    "PR-AUC validation": resultats_val["PR-AUC"],
    "PR-AUC test": resultats_test["PR-AUC"],
    "précision validation": resultats_val["précision"],
    "précision test": resultats_test["précision"],
    "rappel validation": resultats_val["rappel"],
    "rappel test": resultats_test["rappel"],
})
ecart.round(4)

**Lecture.**

Sur la validation, les résultats prolongent ceux de la section 3. Sur le test, ils baissent nettement. La PR-AUC de XGBoost passe de 0,985 à 0,948, celle de la forêt de 0,985 à 0,943, celle de la régression logistique de 0,959 à 0,865. Elles restent toutes très au-dessus du niveau d'un modèle au hasard (0,361), et l'ordre des modèles ne change pas : les deux modèles d'arbres devant la baseline.

**La baisse porte presque entièrement sur la précision.** Pour XGBoost, la précision tombe de 0,93 à 0,69, alors que le rappel reste au niveau de la validation (0,94 puis 0,96). Les modèles continuent donc de reconnaître les attaques, mais ils signalent beaucoup plus de trafic normal : XGBoost lève 7 993 fausses alertes sur 33 662 flux normaux, soit 24 %, contre 7 % sur la validation.

Une partie de la baisse de précision est mécanique : le test contient moins d'attaques (36 % contre 49 %), et à taux de fausses alertes égal, moins d'attaques veut dire une précision plus faible. Mais cela n'explique pas tout, car le taux de fausses alertes, qui ne dépend pas de la proportion d'attaques, a lui-même triplé. Le trafic normal du fichier de test n'est pas reconnu comme celui de l'entraînement. La section 4.4 cherche pourquoi.

Le seuil de la régression logistique (0,10) illustre un autre point : choisi sur la validation pour maximiser le F1, il conduit sur le test à signaler 41 % du trafic normal. Un seuil n'est bon que tant que les données ressemblent à celles sur lesquelles il a été choisi.

### 4.3 Résultats par famille

Part des flux de chaque famille signalés comme attaque, au seuil retenu. Pour les familles d'attaques, c'est le rappel de la famille. Pour « Normal », c'est le taux de fausses alertes.

In [ ]:
pd.concat({nom: E.recall_by_family(fam_test, y_test, scores_test[nom], seuils[nom])["part signalée"]
           for nom in modeles}, axis=1).assign(
    effectif=E.recall_by_family(fam_test, y_test, scores_test["XGBoost"], seuils["XGBoost"])["effectif"]
).round(3)

**Lecture.**

La plupart des familles d'attaques sont détectées à plus de 99 % par les trois modèles. Les **Fuzzers** sont le point faible des modèles d'arbres (83 % pour XGBoost, 89 % pour la forêt), suivis par **Analysis** pour XGBoost (94 %). Les Fuzzers envoient des données aléatoires ou mal formées à un service ; un flux isolé de ce type peut ressembler à une connexion ordinaire, ce qui rend la famille plus difficile à séparer du trafic normal.

Les familles rares (Worms : 43 flux, Backdoor : 70) affichent des taux élevés, mais sur si peu de flux, une ou deux erreurs de plus changeraient le pourcentage de plusieurs points. Ces chiffres sont à lire avec prudence.

La régression logistique a le meilleur rappel dans toutes les familles. Ce n'est pas un signe de qualité : avec son seuil très bas, elle signale aussi 41 % du trafic normal.

### 4.4 D'où viennent les fausses alertes ?

L'écart entre validation et test vient presque entièrement du trafic normal. Nous comparons donc le trafic normal des deux fichiers, et le taux de fausses alertes de XGBoost selon la valeur de `sttl`, la variable déjà repérée en section 2.

In [ ]:
normal_train = train[train[config.TARGET] == 0]
normal_test = test[test[config.TARGET] == 0]
alerte_normal_test = scores_test["XGBoost"][(y_test == 0).to_numpy()] >= seuils["XGBoost"]

par_ttl = pd.concat({
    "part du normal (entraînement)": normal_train["sttl"].value_counts(normalize=True),
    "part du normal (test)": normal_test["sttl"].value_counts(normalize=True),
    "taux de fausses alertes (test)": pd.Series(alerte_normal_test, index=normal_test.index)
                                        .groupby(normal_test["sttl"]).mean(),
}, axis=1).sort_values("part du normal (test)", ascending=False).head(5)
par_ttl.round(3)

In [ ]:
attaques_train = train[train[config.TARGET] == 1]
print("Valeurs de sttl des attaques (entraînement) :",
      attaques_train["sttl"].value_counts(normalize=True).head(3).round(3).to_dict())

fausses = normal_test[alerte_normal_test]
part = fausses["sttl"].isin([254, 62]).mean()
print(f"Fausses alertes de XGBoost sur le test : {len(fausses):,}, dont {part:.2%} avec sttl = 254 ou 62")

**Lecture.**

Le résultat est net.

- Dans le fichier d'entraînement, les attaques ont toutes, à quelques flux près, un `sttl` de 254 (72 %) ou de 62 (28 %).
- Dans le test, les flux normaux avec `sttl` = 31, soit la moitié du trafic normal, ne déclenchent **jamais** de fausse alerte. Ceux avec `sttl` = 254 en déclenchent une fois sur deux (54 %), ceux avec `sttl` = 62 une fois sur quatre (25 %).
- La part du trafic normal ayant `sttl` = 254 double entre l'entraînement et le test (19 % puis 39 %), et celle de `sttl` = 62 triple (4 % puis 11 %).
- La quasi-totalité des fausses alertes de XGBoost concerne des flux avec `sttl` = 254 ou 62.

Tout se passe comme si le modèle avait appris une règle du type « TTL de 254 ou de 62, donc attaque », presque toujours vraie dans le fichier d'entraînement. Quand le fichier de test contient beaucoup plus de trafic normal avec ces mêmes TTL, la règle produit des fausses alertes en masse.

Or le TTL dépend surtout du système d'exploitation de l'émetteur et du nombre de routeurs traversés, pas du caractère malveillant du trafic. Sur le banc d'essai, les machines qui ont généré les attaques avaient vraisemblablement des TTL particuliers. Ce que le modèle exploite ressemble donc à une trace de la configuration du banc d'essai plutôt qu'à un comportement d'attaque : un artefact de collecte.

C'est pour l'instant une corrélation, pas une preuve. Le fait que les erreurs se concentrent sur ces valeurs ne montre pas, à lui seul, que le modèle utilise `sttl` pour décider : d'autres variables liées au TTL ou au banc d'essai pourraient jouer le même rôle. La section 5 le teste directement, avec les mesures d'importance, les explications SHAP et un ré-entraînement sans les variables suspectes. Conformément au protocole, nous ne modifions rien ici à partir du test.

### 4.5 Coût des faux positifs et des faux négatifs

Un faux positif coûte du temps d'analyste. Un faux négatif est une attaque qui passe sans être vue. Pour rendre ces coûts concrets, nous exprimons les résultats du test en charge de travail pour 10 000 flux.

In [ ]:
pd.DataFrame({nom: E.alert_load(y_test, scores_test[nom], seuils[nom]) for nom in modeles}).T.round(0)

Dans le test, 36 % des flux sont des attaques. Sur un vrai réseau, la part est bien plus faible. Le rappel et le taux de fausses alertes ne dépendent pas de cette proportion, mais la précision, si. Le tableau suivant recalcule la précision qu'aurait chaque modèle, avec son rappel et son taux de fausses alertes mesurés sur le test, si les attaques représentaient 10 %, 1 % ou 0,1 % du trafic.

In [ ]:
lignes = {}
for nom in modeles:
    r = resultats_test.loc[nom]
    tfa = r["FP"] / (r["FP"] + r["VN"])
    lignes[nom] = {"taux de fausses alertes": tfa,
                   **{f"précision si {p:.1%} d'attaques": E.precision_at_prevalence(r["rappel"], tfa, p)
                      for p in (0.10, 0.01, 0.001)}}
pd.DataFrame(lignes).T.round(3)

**Lecture.**

**Sur le test**, pour 10 000 flux, XGBoost lève environ 4 965 alertes, dont 1 518 fausses, et laisse passer 158 attaques. La forêt fait l'arbitrage inverse : plus de fausses alertes (1 729) mais moins d'attaques manquées (96). La régression logistique ne manque presque rien (6) au prix de 2 625 fausses alertes. Aucun de ces choix n'est bon ou mauvais en soi ; il dépend de ce que l'équipe peut absorber.

**Sur un vrai réseau, la situation serait bien pire.** Avec le taux de fausses alertes mesuré sur le test, si les attaques représentaient 1 % du trafic, moins de 4 % des alertes de XGBoost seraient de vraies attaques : 96 alertes sur 100 seraient fausses. À 0,1 %, la précision tomberait sous 0,5 %. Même avec le taux de fausses alertes plus favorable de la validation (7 %), la précision à 1 % d'attaques ne dépasserait pas environ 12 %. Un détecteur exploitable dans un centre opérationnel de sécurité doit viser un taux de fausses alertes de l'ordre du millième ou moins.

**Faux positifs contre faux négatifs.** Un faux négatif peut coûter beaucoup plus cher qu'un faux positif : une intrusion non détectée peut mener à une fuite de données ou à un chiffrement des systèmes, alors qu'une fausse alerte coûte quelques minutes d'analyste. Cela pousse à privilégier le rappel. Mais ce raisonnement a une limite : quand les fausses alertes se comptent par milliers, les analystes saturent (fatigue d'alerte), traitent les alertes plus vite et moins bien, et finissent par écarter la vraie attaque avec les autres. Au-delà d'un certain volume, les faux positifs créent eux-mêmes des faux négatifs.

Le F1, que nous avons utilisé pour choisir le seuil, donne le même poids aux deux types d'erreur. En production, le seuil se choisirait plutôt avec l'équipe, par exemple à partir d'un budget d'alertes par jour qu'elle peut traiter, et sur des données récentes du réseau surveillé. La courbe précision-rappel du test montre qu'un seuil plus élevé réduirait fortement les fausses alertes au prix d'une partie des attaques ; nous ne le réglons pas ici, puisque le test ne doit servir à aucun réglage.

## 5. Interprétabilité

Les sections précédentes ont mesuré **à quel point** les modèles détectent les attaques. Celle-ci cherche **sur quoi** ils s'appuient. La section 4 a donné une piste : les fausses alertes du test se concentrent sur deux valeurs de `sttl`. Il s'agit maintenant de vérifier ce que le modèle utilise réellement.

**Modèle étudié.** Nous expliquons la forêt aléatoire. Le cahier des charges demande de comparer l'importance interne de la forêt à deux autres mesures ; calculer les trois sur le même modèle rend la comparaison équitable. Les performances de la forêt et de XGBoost sont très proches (sections 3 et 4), donc ce choix ne sacrifie rien.

**Variables d'origine.** Le modèle voit 192 colonnes à cause de l'encodage one-hot. Toutes les mesures sont ramenées aux 42 variables d'origine : pour l'importance interne et SHAP, en additionnant les colonnes one-hot de chaque variable textuelle (les deux mesures sont additives) ; la permutation mélange directement les variables d'origine, avant le pipeline.

### 5.1 Trois mesures d'importance

1. **Importance interne de la forêt** (diminution moyenne de l'impureté). À chaque division d'un arbre sur une variable, les données deviennent plus homogènes ; la somme de ces gains sur tous les arbres donne l'importance. Elle est gratuite, mais calculée sur les données d'entraînement, et elle favorise les variables qui offrent beaucoup de points de coupure possibles.
2. **Importance par permutation sur le test.** On mélange les valeurs d'une variable entre les flux du test et on mesure la baisse de PR-AUC (cinq répétitions). Elle porte sur des données jamais vues, mais quand deux variables portent la même information, mélanger l'une laisse l'autre compenser.
3. **Valeur SHAP absolue moyenne** sur un échantillon de 2 000 flux du test, tiré au hasard et stratifié par famille d'attaque (limite du cahier des charges). Les valeurs SHAP de la forêt sont en points de probabilité d'attaque.

Les deux calculs les plus longs (permutation et SHAP, environ une minute chacun) sont mis en cache comme les recherches de la section 3.

In [ ]:
foret = modeles["Forêt aléatoire"]

imp_interne = X.forest_importance(foret)
imp_permutation = X.permutation_importance_cached("random_forest", foret, X_test, y_test)
X_shap, y_shap, fam_shap = X.shap_sample(X_test, y_test, fam_test)
valeurs_shap, base_shap, colonnes_shap = X.shap_values_cached("random_forest", foret, X_shap)
imp_shap = X.mean_abs_shap(valeurs_shap, foret)

# Contrôle : pour chaque flux, valeur de base + somme des valeurs SHAP = score du modèle.
ecart_max = np.abs(base_shap + valeurs_shap.sum(axis=1) - foret.predict_proba(X_shap)[:, 1]).max()
print(f"Valeur de base SHAP (score moyen) : {base_shap:.3f}")
print(f"Écart maximal entre score et somme SHAP : {ecart_max:.1e}")

In [ ]:
mesures = {
    "interne (forêt)": imp_interne,
    "permutation (test)": imp_permutation["baisse moyenne"],
    "SHAP |moyenne| (test)": imp_shap,
}
X.plot_measures(mesures, top=10, filename="trois_mesures_importance.png")

#### Tableau comparatif

Valeurs des trois mesures pour les variables présentes dans le top 10 d'au moins une mesure, puis leur rang.

In [ ]:
top = X.ranking_table(mesures, top=10).index
pd.DataFrame({
    "interne": imp_interne,
    "permutation": imp_permutation["baisse moyenne"],
    "SHAP": imp_shap,
}).loc[top].round(4)

In [ ]:
X.ranking_table(mesures, top=10)

In [ ]:
print("Corrélation de Spearman entre les classements (42 variables) :")
X.rank_correlations(mesures).round(3)

In [ ]:
negatives = imp_permutation[imp_permutation["baisse moyenne"] < 0]
print(f"Variables dont la permutation améliore légèrement le score : {len(negatives)} "
      f"(baisse la plus négative : {negatives['baisse moyenne'].min():.4f})")

#### Variables corrélées

Plusieurs variables importantes sont liées entre elles. `ct_state_ttl` est calculée par les auteurs à partir de l'état de la connexion et des TTL. Le tableau suivant donne les corrélations de rang (Spearman) entre les variables du haut du classement, sur l'entraînement.

In [ ]:
liees = ["sttl", "ct_state_ttl", "dttl", "dload", "dmean", "dbytes", "ct_srv_dst", "ct_dst_src_ltm"]
X_train[liees].corr(method="spearman").round(2)

Quand des variables portent la même information, la permutation d'une seule sous-estime leur poids : les autres compensent. Pour mesurer le poids réel des variables liées au TTL, nous les mélangeons **ensemble** (même permutation pour les trois colonnes).

In [ ]:
groupe_ttl = ["sttl", "ct_state_ttl", "dttl"]
baisse, ecart_type, reference = X.group_permutation_drop(foret, X_test, y_test, groupe_ttl)
print(f"PR-AUC de la forêt sur le test : {reference:.3f}")
print(f"Baisse quand {groupe_ttl} sont mélangées ensemble : {baisse:.3f} ± {ecart_type:.3f}")
print(f"Somme des baisses individuelles : {imp_permutation.loc[groupe_ttl, 'baisse moyenne'].sum():.3f}")

part_shap_ttl = imp_shap[groupe_ttl].sum() / imp_shap.sum()
part_interne_ttl = imp_interne[groupe_ttl].sum()
print(f"Part des trois variables TTL : {part_interne_ttl:.1%} de l'importance interne, "
      f"{part_shap_ttl:.1%} de la somme des |SHAP| moyens")

#### Sens des effets (SHAP)

Le graphique suivant montre, pour chaque flux de l'échantillon, la valeur SHAP des colonnes les plus influentes. La couleur indique la valeur de la variable (rouge : élevée, bleu : faible). Un point à droite pousse le score vers « attaque », un point à gauche vers « normal ». Les variables textuelles apparaissent ici colonne par colonne (par exemple `service_-`).

In [ ]:
import shap
Z_shap = pd.DataFrame(foret.named_steps["prep"].transform(X_shap), columns=colonnes_shap)
shap.summary_plot(valeurs_shap, Z_shap, max_display=12, show=False)
plt.title("Valeurs SHAP de la forêt, 2 000 flux du test")
plt.tight_layout()
plt.savefig(config.FIGURES / "shap_beeswarm_foret.png", dpi=150, bbox_inches="tight")
plt.show()

**Lecture.**

**Les trois mesures s'accordent sur l'essentiel.** `sttl` est première pour les trois, `ct_state_ttl` deuxième pour les trois, et `dttl` est dans le top 7 de chacune. Sur l'ensemble des 42 variables, les classements sont fortement corrélés (Spearman de 0,75 à 0,90). Les variables liées au TTL représentent à elles seules 62 % de l'importance interne de la forêt et 59 % de la somme des valeurs SHAP absolues moyennes.

**Le poids du TTL, mesuré directement.** Mélanger `sttl` seule fait baisser la PR-AUC du test de 0,17. Mélanger ensemble `sttl`, `ct_state_ttl` et `dttl` la fait baisser de 0,29, de 0,943 à environ 0,65. C'est plus que la somme des baisses individuelles (0,23) : quand une seule de ces variables est mélangée, les deux autres compensent en partie, ce qui illustre le piège des variables corrélées signalé dans le cahier des charges. Le niveau d'un modèle au hasard étant de 0,361, les trois variables TTL portent à elles seules environ la moitié de l'avance de la forêt sur le hasard.

**Le sens des effets.** Le graphique SHAP montre une séparation nette : un `sttl` faible (31, en bleu à gauche) fait baisser la probabilité d'attaque d'environ 0,3 ; un `sttl` élevé (254, en rouge à droite) l'augmente d'environ 0,2. Rapporté à une valeur de base de 0,49, une seule variable suffit presque à faire basculer la décision. C'est exactement le mécanisme qui produisait les fausses alertes de la section 4.4 sur les flux normaux à `sttl` = 254.

**Pourquoi les classements diffèrent-ils ailleurs ?** Trois effets expliquent l'essentiel des écarts.

- *Données différentes.* L'importance interne est calculée sur l'entraînement, la permutation et SHAP sur le test. `dbytes` est 17e pour l'importance interne mais 3e pour la permutation : l'importance interne mesure combien la forêt s'en est servie pour séparer les données d'entraînement, la permutation mesure ce que la variable apporte sur des données nouvelles, et les deux n'ont pas de raison de coïncider, surtout quand le trafic du test diffère de celui de l'entraînement.
- *Biais de l'importance interne.* Elle favorise les variables qui offrent beaucoup de points de coupure. `service`, variable textuelle à 13 valeurs, est 13e en interne mais 4e en permutation et 6e avec SHAP.
- *Variables corrélées.* `dload` est 4e en interne mais 9e et 10e ailleurs. Elle est fortement corrélée à `sttl` (Spearman −0,81) et à `ct_state_ttl` (−0,85) : la forêt l'utilise dans certaines divisions, mais l'information qu'elle porte est déjà presque entièrement dans les variables TTL. La mélanger seule coûte peu, et SHAP lui attribue aussi une faible contribution.

Les six permutations qui améliorent légèrement le score (au plus 0,002) correspondent à des variables sans effet utile ; l'écart est du bruit et se lit comme une importance nulle.

**Ce qui reste à établir.** Ces mesures montrent que la forêt s'appuie massivement sur le TTL. Elles ne disent pas encore si ce choix est légitime du point de vue réseau, ni ce que vaudrait le modèle sans ces variables. C'est l'objet de l'audit (5.3) et de l'ablation (5.4).

### 5.2 Cinq alertes expliquées

Le cahier des charges demande d'expliquer au moins cinq flux du test avec SHAP : deux vrais positifs (VP), deux faux positifs (FP) et un faux négatif (FN), et de rédiger pour chacun ce qu'écrirait un analyste. Nous utilisons la forêt, au seuil choisi sur la validation, et les valeurs SHAP déjà calculées sur l'échantillon de 2 000 flux du test.

**Choix des cas.** Pour ne pas choisir des exemples à la main, les règles sont fixées à l'avance et le tirage se fait au hasard avec la graine du projet :

- deux VP de deux familles différentes, les deux familles les plus fréquentes parmi les VP de l'échantillon ;
- deux FP, l'un avec `sttl` = 254 et l'autre avec `sttl` = 62, les deux groupes qui produisent les fausses alertes (section 4.4) ;
- un FN, tiré parmi les FN de l'échantillon.

**Lecture d'un graphique en cascade.** On part en bas de la valeur de base, le score moyen du modèle (environ 0,49). Chaque barre ajoute la contribution d'une variable, en rouge vers « attaque », en bleu vers « normal », jusqu'au score final du flux en haut. La valeur de la variable pour ce flux est écrite à gauche, dans ses unités d'origine.

In [ ]:
seuil_foret = seuils["Forêt aléatoire"]
scores_shap = foret.predict_proba(X_shap)[:, 1]
cas, issue_shap = X.pick_cases(X_shap, y_shap, fam_shap, scores_shap, seuil_foret)
noms_cas = list(cas)

print("Répartition dans l'échantillon :", pd.Series(issue_shap).value_counts().to_dict())
pd.DataFrame({
    "famille réelle": [fam_shap.loc[i] for i in cas.values()],
    "score": [scores_shap[X_shap.index.get_loc(i)] for i in cas.values()],
}, index=noms_cas).round(3)

In [ ]:
nom = noms_cas[0]
pos = X_shap.index.get_loc(cas[nom])
expl = X.explanation_for(pos, valeurs_shap, base_shap, foret, X_shap)
print(f"{nom} : famille réelle = {fam_shap.iloc[pos]}, score = {scores_shap[pos]:.3f} (seuil {seuil_foret:.3f})")
X.plot_waterfall(expl, nom, "alerte_vp1.png")

**Note d'analyste, VP 1 (Exploits).** Connexion HTTP de 2 secondes, 10 paquets dans chaque sens, environ 830 octets envoyés et 1 360 reçus, fermée normalement (`FIN`). Score 0,90 : alerte forte.

Ce qui a déclenché l'alerte : presque entièrement les variables TTL. `sttl` = 62 ajoute 0,18, `ct_state_ttl` et `dttl` encore 0,13 à elles deux. Viennent ensuite des compteurs de connexions faibles (`ct_srv_dst` = 1, `ct_dst_src_ltm` = 1 : premier échange de ce type entre ces machines récemment). Rien dans les volumes ou la durée ne signale quelque chose d'anormal ; `dbytes` tire même légèrement vers « normal ».

Avis : l'alerte est juste, mais pour une raison que je ne peux pas vérifier. Un TTL de 62 correspond typiquement à un système qui part de 64 (Linux, par exemple) et traverse deux routeurs : il renseigne sur la machine et le chemin, pas sur une attaque. Le flux lui-même ressemble à une requête web ordinaire. Suite à donner : récupérer la capture du contenu HTTP (URL, en-têtes, corps de la requête) ; c'est là que se trouverait la trace de l'exploit, et le modèle ne la voit pas.

In [ ]:
nom = noms_cas[1]
pos = X_shap.index.get_loc(cas[nom])
expl = X.explanation_for(pos, valeurs_shap, base_shap, foret, X_shap)
print(f"{nom} : famille réelle = {fam_shap.iloc[pos]}, score = {scores_shap[pos]:.3f} (seuil {seuil_foret:.3f})")
X.plot_waterfall(expl, nom, "alerte_vp2.png")

**Note d'analyste, VP 2 (Fuzzers).** Connexion TCP d'une seconde sans service reconnu (`service` = `-`), 10 paquets envoyés et 6 reçus, 534 octets envoyés et 268 reçus. Score 0,56, juste au-dessus du seuil (0,48) : alerte faible.

Ce qui a déclenché l'alerte : `sttl` = 254 (+0,19), puis `ct_dst_src_ltm` = 1 (+0,09), `dttl` et `ct_state_ttl`. Les petits volumes (`smean` = 53, `sbytes` = 534) tirent vers « normal » (−0,16 à eux deux) et ramènent le score près du seuil.

Avis : sans le TTL, ce flux serait sous le seuil. L'alerte repose sur la même variable que la précédente, et rien dans le flux ne décrit un comportement de fuzzing (envoi répété de données mal formées). Un fuzzer se repère plutôt sur une série de connexions vers le même service. Suite à donner : regarder les autres connexions de la même source vers la même destination dans la minute, et le contenu envoyé.

In [ ]:
nom = noms_cas[2]
pos = X_shap.index.get_loc(cas[nom])
expl = X.explanation_for(pos, valeurs_shap, base_shap, foret, X_shap)
print(f"{nom} : famille réelle = {fam_shap.iloc[pos]}, score = {scores_shap[pos]:.3f} (seuil {seuil_foret:.3f})")
X.plot_waterfall(expl, nom, "alerte_fp1.png")

**Note d'analyste, FP 1 (normal, `sttl` = 254).** Connexion TCP sans service reconnu, 10 paquets envoyés et 6 reçus, 2 516 octets envoyés et 268 reçus, une demi-seconde. Score 0,65 : alerte, à tort.

Ce qui a déclenché l'alerte : `sttl` = 254 (+0,19), complété par `dttl` et `ct_state_ttl` (+0,09). Seul `ct_srv_dst` = 3 tire nettement vers « normal ».

Avis : faux positif. Le profil du flux est presque identique à celui du VP 2 (même nombre de paquets, mêmes 268 octets reçus, mêmes TTL) ; il envoie seulement plus de données. Le modèle n'a aucun moyen de les distinguer autrement que par de petites différences de volume. C'est le schéma qui produit les milliers de fausses alertes du test : un TTL de 254 sur du trafic normal suffit à faire franchir le seuil. Suite à donner : classer sans suite si la source est un serveur connu ; signaler que la règle « TTL 254 » génère du bruit.

In [ ]:
nom = noms_cas[3]
pos = X_shap.index.get_loc(cas[nom])
expl = X.explanation_for(pos, valeurs_shap, base_shap, foret, X_shap)
print(f"{nom} : famille réelle = {fam_shap.iloc[pos]}, score = {scores_shap[pos]:.3f} (seuil {seuil_foret:.3f})")
X.plot_waterfall(expl, nom, "alerte_fp2.png")

**Note d'analyste, FP 2 (normal, `sttl` = 62).** Connexion HTTP d'une seconde, 10 paquets dans chaque sens, 810 octets envoyés et 1 160 reçus, fermée normalement. Score 0,88 : alerte forte, à tort.

Ce qui a déclenché l'alerte : exactement les mêmes variables, dans le même ordre et avec presque les mêmes valeurs que pour le VP 1. `sttl` = 62 (+0,19), `ct_state_ttl` (+0,08), `dttl` (+0,06), `ct_srv_dst` = 1 (+0,05).

Avis : faux positif, et le plus instructif des cinq. Ce flux normal et l'exploit du VP 1 sont presque jumeaux : même protocole, même service, même nombre de paquets, volumes très proches, mêmes TTL. Pour le modèle, ce sont le même flux. Pour un analyste aussi, tant qu'il n'a que ces statistiques. La différence entre une requête web légitime et une requête qui exploite une faille est dans le contenu, que les variables de flux ne décrivent pas. Suite à donner : classer sans suite après vérification du contenu ; noter que, sur ce type de trafic, les statistiques de flux ne suffisent pas à trancher.

In [ ]:
nom = noms_cas[4]
pos = X_shap.index.get_loc(cas[nom])
expl = X.explanation_for(pos, valeurs_shap, base_shap, foret, X_shap)
print(f"{nom} : famille réelle = {fam_shap.iloc[pos]}, score = {scores_shap[pos]:.3f} (seuil {seuil_foret:.3f})")
X.plot_waterfall(expl, nom, "alerte_fn.png")

**Note d'analyste, FN (Fuzzers manqué).** Connexion TCP sans service reconnu, 10 paquets envoyés et 6 reçus, 484 octets envoyés et 268 reçus, 0,3 seconde. Score 0,36 : pas d'alerte, alors que c'était une attaque.

Ce qui a fait baisser le score : `ct_srv_dst` = 13 (−0,14), puis `ct_dst_src_ltm` = 5 et `ct_srv_src` = 13 (−0,05 chacun). `sttl` = 254 poussait pourtant vers « attaque » (+0,16).

Avis : le flux a le même profil que le VP 2 et le FP 1. Ce qui le fait passer sous le seuil, ce sont les compteurs de connexions élevés : 13 connexions récentes vers le même service. Le modèle a appris qu'un service très sollicité est plutôt normal, ce qui est vrai en moyenne (un serveur actif), mais c'est précisément ce que produit un fuzzer qui répète ses essais. Une même valeur peut donc signifier « serveur occupé » ou « attaque répétée » selon le contexte. Suite à donner : pour ce type d'attaque, une règle sur la répétition des connexions d'une même source vers un même service serait plus parlante qu'un score par flux isolé.

#### Les cinq flux côte à côte

In [ ]:
colonnes = ["proto", "service", "state", "dur", "spkts", "dpkts", "sbytes", "dbytes",
            "sttl", "dttl", "ct_state_ttl", "ct_srv_dst", "ct_dst_src_ltm"]
cote_a_cote = X_shap.loc[list(cas.values()), colonnes].copy()
cote_a_cote.index = noms_cas
cote_a_cote.insert(0, "score", [scores_shap[X_shap.index.get_loc(i)] for i in cas.values()])
cote_a_cote.round(3)

**Ce que montrent les cinq explications ensemble.**

**Le TTL domine chaque décision, quelle que soit la réponse.** Dans les cinq cas, `sttl` est la plus forte contribution vers « attaque », entre +0,16 et +0,19, que le flux soit une attaque ou du trafic normal. Une variable qui pousse de la même façon pour une attaque et pour un flux normal ne reconnaît pas l'attaque : elle reconnaît le type de machine émettrice.

**Les erreurs viennent de flux indiscernables.** Le VP 1 (exploit) et le FP 2 (normal) sont presque identiques. Le VP 2 (fuzzer détecté), le FP 1 (normal signalé) et le FN (fuzzer manqué) aussi. Ce n'est pas un défaut de réglage : ces flux ont les mêmes statistiques, et la décision se joue sur de petites différences de volume ou de compteurs. C'est le chevauchement de classes décrit par Zoghi et Serpen (2021), vu flux par flux.

**Les explications ne seraient pas exploitables en centre opérationnel.** Un analyste qui reçoit « alerte parce que TTL = 62 » ne peut ni confirmer ni infirmer l'alerte avec cette information : il doit retourner au contenu du trafic. L'explicabilité joue ici pleinement son rôle, mais pas celui qu'on attendait : elle ne justifie pas les alertes, elle révèle qu'elles reposent sur un indice sans rapport avec l'attaque.

Ces observations portent sur cinq flux. La section 5.3 examine la plausibilité des dix variables principales, et la section 5.4 mesure ce que vaut le modèle sans les variables suspectes.

### 5.3 Audit de plausibilité des dix variables principales

Pour chacune des dix variables les plus importantes selon SHAP (section 5.1), la question est simple : un analyste réseau accepterait-il cette variable comme raison d'une alerte ? Nous classons chaque variable en trois catégories :

- **plausible** : elle décrit un comportement qui peut réellement distinguer une attaque (volumes, répétition des connexions, service visé) ;
- **à surveiller** : elle décrit le trafic, mais elle est fortement liée à une variable suspecte ou dépend des conditions de capture ;
- **suspecte** : elle décrit la machine ou le réseau du banc d'essai plutôt que le comportement, et ne devrait pas permettre de reconnaître une attaque ailleurs.

In [ ]:
top10 = imp_shap.index[:10]
pd.DataFrame({
    "rang SHAP": range(1, 11),
    "rang interne": [int(imp_interne.rank(ascending=False)[f]) for f in top10],
    "rang permutation": [int(imp_permutation["baisse moyenne"].rank(ascending=False)[f]) for f in top10],
}, index=top10)

#### Les éléments de preuve sur le TTL

In [ ]:
# Valeurs de sttl et dttl dans l'entraînement, avec la part d'attaques.
for col in ["sttl", "dttl"]:
    t = pd.crosstab(train[col], train[config.TARGET])
    t.columns = ["normal", "attaque"]
    t["part d'attaques"] = (t["attaque"] / t.sum(axis=1)).round(3)
    display(t.sort_values("normal", ascending=False).head(5))

In [ ]:
# Quelles valeurs de sttl chaque famille utilise-t-elle ?
pd.crosstab(train[config.ATTACK_CAT], train["sttl"])[[31, 62, 254]]

In [ ]:
# ct_state_ttl est construite à partir des TTL : valeurs de sttl et dttl observées pour chaque valeur.
train.groupby("ct_state_ttl")[["sttl", "dttl"]].agg(lambda s: sorted(s.unique()))

**Tableau d'audit.**

| Variable | Ce qu'elle mesure | Verdict | Raison |
|---|---|---|---|
| `sttl` | TTL des paquets de la source | **suspecte** | Fixé par le système d'exploitation de l'émetteur et diminué à chaque routeur. Il identifie la machine et sa position, pas l'attaque. Dans l'entraînement, aucune attaque n'a `sttl` = 31 et toutes les familles d'attaques utilisent seulement 62 ou 254. |
| `ct_state_ttl` | Valeur calculée par les auteurs à partir de l'état et des plages de TTL | **suspecte** | Elle recode directement `sttl` et `dttl` (par exemple, la valeur 1 correspond exactement à `sttl` de 62 ou 254 avec `dttl` = 252). Elle hérite de l'artefact. |
| `dttl` | TTL des paquets de la destination | **suspecte** | Même logique, côté machine qui répond. Les 31 283 flux avec `sttl` = 31 ont tous `dttl` = 29 : c'est une signature de paire de machines. |
| `ct_srv_dst` | Connexions récentes vers le même service et la même destination (sur les 100 dernières) | à surveiller | Plausible (balayages, attaques répétées), mais le compte dépend du rythme de génération du trafic sur le banc d'essai. Le FN de la section 5.2 montre aussi qu'une valeur élevée peut signifier « serveur actif » comme « attaque répétée ». |
| `ct_dst_src_ltm` | Connexions récentes entre la même source et la même destination | à surveiller | Même remarque que la précédente. |
| `service` | Protocole applicatif (http, dns, ftp...) | plausible | Le service visé fait partie du contexte normal d'une alerte. |
| `dmean` | Taille moyenne des paquets de la destination | plausible | Une réponse inhabituelle peut signaler un échange anormal. Corrélée au TTL (Spearman −0,69). |
| `ct_srv_src` | Connexions récentes de la même source vers le même service | à surveiller | Comme `ct_srv_dst`. |
| `dbytes` | Octets envoyés par la destination | plausible | Volume de la réponse, information classique en détection. |
| `dload` | Débit de la destination (bits par seconde) | à surveiller | Calculé à partir des octets et de la durée, donc plausible, mais très corrélé au TTL (Spearman −0,81). Il peut servir de relais au même artefact. |

**Variables suspectes retenues pour l'ablation :** `sttl`, `dttl` et `ct_state_ttl`. Le cahier des charges signale aussi le risque des variables identifiantes (adresses, ports, horodatage) : elles sont absentes des fichiers de partition, et `id` a été retirée en section 2.

### 5.4 Ablation des variables suspectes

Nous ré-entraînons les trois modèles **sans** `sttl`, `dttl` et `ct_state_ttl`, en gardant les hyperparamètres trouvés en section 3 : la seule différence avec les modèles initiaux est l'ensemble des variables. Le seuil de chaque modèle est de nouveau choisi sur la validation, avec la même règle (F1 maximal). Les modèles ré-entraînés sont mis en cache.

In [ ]:
TTL = ["sttl", "dttl", "ct_state_ttl"]

def evaluer(modeles_, retirees):
    """Seuil choisi sur la validation, puis résultats sur la validation et le test."""
    lignes, scores = {}, {}
    normal_test = (y_test == 0).to_numpy()
    for nom, m in modeles_.items():
        sv = m.predict_proba(X_val.drop(columns=retirees))[:, 1]
        seuil, _ = E.best_f1_threshold(y_val, sv)
        st = m.predict_proba(X_test.drop(columns=retirees))[:, 1]
        rv, rt = E.summary(y_val, sv, seuil), E.summary(y_test, st, seuil)
        alerte = st >= seuil
        fa_ttl = pd.Series(alerte[normal_test]).groupby(X_test["sttl"].to_numpy()[normal_test]).mean()
        lignes[nom] = {
            "PR-AUC validation": rv["PR-AUC"], "PR-AUC test": rt["PR-AUC"],
            "précision test": rt["précision"], "rappel test": rt["rappel"], "F1 test": rt["F1"],
            "fausses alertes test": rt["FP"] / (rt["FP"] + rt["VN"]),
            "fausses alertes, sttl = 31": fa_ttl.get(31), "fausses alertes, sttl = 254": fa_ttl.get(254),
            "fausses alertes, sttl = 62": fa_ttl.get(62),
        }
        scores[nom] = (st, seuil)
    return pd.DataFrame(lignes).T, scores

sans_ttl = {nom: M.refit_without_cached(nom, recherches[nom].best_params_, X_train, y_train, TTL, "ttl")
            for nom in modeles}
res_avec, _ = evaluer(modeles, [])
res_sans, scores_sans = evaluer(sans_ttl, TTL)
pd.concat({"avec TTL": res_avec, "sans TTL": res_sans}).round(3)

**Lecture.**

**Les modèles d'arbres ne perdent presque rien.** Sans les trois variables TTL, la PR-AUC de validation de la forêt et de XGBoost reste à 0,985. Sur le test, elle baisse de 0,005 seulement (0,943 puis 0,938 pour la forêt, 0,948 puis 0,943 pour XGBoost). Précision, rappel et F1 bougent de un à deux points.

**La régression logistique, elle, perd beaucoup** : sa PR-AUC de test passe de 0,865 à 0,720. Elle s'appuyait sur le TTL, et un modèle linéaire ne peut pas reconstruire cette information en combinant d'autres variables.

**Mais les fausses alertes ont exactement la même forme.** Le modèle sans TTL ne voit plus `sttl`, et pourtant :

- les flux normaux avec `sttl` = 31 ne sont presque jamais signalés (0,2 % pour la forêt) ;
- ceux avec `sttl` = 254 le sont toujours dans 59 à 66 % des cas, un peu plus qu'avant ;
- ceux avec `sttl` = 62 dans 28 à 29 % des cas.

Si le modèle avait appris à reconnaître des attaques, retirer le TTL aurait dû faire disparaître ce lien entre la valeur du TTL et les erreurs. Il est intact. Le modèle continue de séparer le trafic selon le groupe de machines qui l'a émis, sans avoir besoin des colonnes TTL pour le faire.

#### Le TTL est-il encore présent sans ses colonnes ?

Si le modèle sans TTL continue à séparer les flux normaux selon leur `sttl`, c'est que les autres variables portent la même information. Nous le testons directement. Sur le **trafic normal seulement** (l'étiquette d'attaque ne joue donc aucun rôle), nous entraînons un petit modèle XGBoost à reconnaître si un flux a un `sttl` de 62 ou 254 (les valeurs des machines d'attaque) ou une autre valeur, **sans les trois variables TTL**. Il est entraîné sur l'entraînement et évalué sur le test.

In [ ]:
auc_hote, acc_hote, imp_hote = X.host_group_probe(train, test, TTL)
print(f"Sans les variables TTL : ROC-AUC = {auc_hote:.3f}, exactitude équilibrée = {acc_hote:.3f}")
print("Variables les plus utilisées pour reconnaître le groupe de machines :")
imp_hote.head(8).round(3)

In [ ]:
# Deuxième niveau d'ablation : on retire aussi les délais de la poignée de main TCP.
DELAIS = ["tcprtt", "synack", "ackdat"]
auc_hote2, acc_hote2, _ = X.host_group_probe(train, test, TTL + DELAIS)
print(f"Sans TTL ni délais TCP : ROC-AUC = {auc_hote2:.3f}, exactitude équilibrée = {acc_hote2:.3f}")

sans_ttl_delais = {nom: M.refit_without_cached(nom, recherches[nom].best_params_, X_train, y_train,
                                               TTL + DELAIS, "ttl_timing")
                   for nom in modeles}
res_sans2, _ = evaluer(sans_ttl_delais, TTL + DELAIS)
res_sans2.round(3)

#### Sur quoi s'appuie la forêt sans TTL ?

In [ ]:
foret_sans = sans_ttl["Forêt aléatoire"]
X_shap_sans = X_shap.drop(columns=TTL)
valeurs_sans, _, _ = X.shap_values_cached("random_forest_sans_ttl", foret_sans, X_shap_sans)
X.mean_abs_shap(valeurs_sans, foret_sans).head(10).round(4).rename("SHAP |moyenne|, sans TTL")

**Lecture.**

**L'identité des machines est partout.** Sans aucune variable TTL, un modèle entraîné sur le seul trafic normal reconnaît sur le test si un flux vient des machines « 62/254 » ou des autres avec une ROC-AUC de 1,000. Il s'appuie surtout sur `dload`, le débit de la destination (68 % de son importance), déjà classée « à surveiller » dans l'audit pour sa corrélation avec le TTL. En retirant aussi les trois délais de la poignée de main TCP, la ROC-AUC reste à 1,000, et les trois détecteurs se comportent comme avant (PR-AUC de test de 0,934 et 0,939 pour les arbres, même répartition des fausses alertes).

**La forêt sans TTL s'appuie sur les relais de la même information.** Ses variables principales deviennent `ackdat`, `dload` et `tcprtt` : les délais de la poignée de main TCP et le débit, qui dépendent eux aussi des machines et de leur position sur le réseau du banc d'essai.

**Le modèle initial était-il robuste ?** Ses scores, oui : ils résistent au retrait des variables suspectes. Mais cette robustesse ne prouve pas qu'il a appris des comportements d'attaque. Elle montre que l'artefact n'est pas une colonne qu'on pourrait retirer : c'est la disposition du banc d'essai elle-même. Dans UNSW-NB15, les attaques viennent d'un ensemble précis de machines, et tout ce qui caractérise ces machines (TTL, délais réseau, débit) permet de les reconnaître. Retirer des colonnes déplace le raccourci vers d'autres colonnes, sans le supprimer.

Le cahier des charges demande de vérifier si le modèle initial était robuste. Notre réponse est double : il est robuste à l'ablation, mais il n'est pas robuste à un changement du trafic normal. La section 4 l'a montré : dès que le fichier de test contient plus de trafic normal venant des machines « 62/254 », le taux de fausses alertes triple. Ce second type de robustesse est celui qui compte pour un déploiement réel.

### 5.5 Stabilité des explications

Une explication digne de confiance ne doit pas changer quand on modifie quelque chose qui n'a rien à voir avec le problème. Nous vérifions deux sources de variation, séparément :

- **la graine** : même découpage entraînement / validation, forêt entraînée avec quatre autres graines (1 à 4) ;
- **le découpage** : quatre autres découpages entraînement / validation (graines de découpage 1 à 4), forêt entraînée avec la graine 42.

Chaque variante garde les hyperparamètres de la section 3. Les valeurs SHAP sont toujours calculées sur **les mêmes 2 000 flux du test** qu'en section 5.1 : les écarts observés viennent donc du modèle, pas des flux expliqués. La référence (graine 42, découpage 42) est recalculée de la même façon, ce qui permet de vérifier qu'elle reproduit exactement la section 5.1.

Les neuf entraînements et calculs SHAP prennent environ un quart d'heure la première fois ; ils sont mis en cache.

In [ ]:
train_complet = train_clean  # fichier d'entraînement nettoyé, avant découpage
params_foret = recherches["Forêt aléatoire"].best_params_

variantes = {"référence": S.run(train_complet, X_test, y_test, X_shap, params_foret, 42, 42)}
for g in [1, 2, 3, 4]:
    variantes[f"graine {g}"] = S.run(train_complet, X_test, y_test, X_shap, params_foret, 42, g)
for g in [1, 2, 3, 4]:
    variantes[f"découpage {g}"] = S.run(train_complet, X_test, y_test, X_shap, params_foret, g, 42)

ecart_ref = (variantes["référence"]["shap"].abs().mean() - imp_shap).abs().max()
print(f"Écart maximal entre la référence recalculée et la section 5.1 : {ecart_ref:.1e}")

#### Classement global

In [ ]:
S.summarise(variantes, variantes["référence"]).round(3)

In [ ]:
S.top_features(variantes)

#### Explications locales

Pour chacun des cinq flux de la section 5.2 : la variable qui contribue le plus, et la contribution de `sttl`, dans chacune des neuf variantes.

In [ ]:
stabilite_cas = S.case_stability(variantes, cas)
pd.DataFrame({
    "variable principale (9 variantes)": stabilite_cas.groupby("cas")["variable principale"]
                                          .agg(lambda s: ", ".join(f"{k} × {v}" for k, v in s.value_counts().items())),
    "contribution de sttl, min": stabilite_cas.groupby("cas")["contribution de sttl"].min(),
    "contribution de sttl, max": stabilite_cas.groupby("cas")["contribution de sttl"].max(),
}).loc[noms_cas].round(3)

**Lecture.**

**Le haut du classement ne bouge pas.** Dans les neuf variantes, `sttl` est première et `ct_state_ttl` deuxième. Les places 3 à 5 sont toujours occupées par les trois mêmes variables (`ct_srv_dst`, `dttl`, `ct_dst_src_ltm`), qui échangent parfois leur ordre. La part des variables TTL dans les valeurs SHAP reste entre 56 % et 59 %. La référence recalculée reproduit exactement la section 5.1.

**Le bas du top 10 varie, mais dans un groupe restreint.** Les places 6 à 10 tournent entre `service`, `dload`, `dbytes`, `dmean`, `ct_srv_src`, `smean` et `sbytes`. Huit à dix des dix premières variables sont communes avec la référence, et la corrélation de rang sur les 42 variables reste au moins à 0,97 pour SHAP comme pour l'importance interne. Ces variables ont des importances proches et sont corrélées entre elles ; un petit changement du modèle suffit à intervertir leur ordre. Il ne faut donc pas tirer de conclusion de la place exacte d'une variable au-delà du top 5.

**Les explications locales sont stables aussi.** Pour les cinq flux de la section 5.2, `sttl` est la variable principale dans les neuf variantes, soit 45 fois sur 45, avec une contribution comprise entre +0,16 et +0,22.

**Les performances aussi.** La PR-AUC de validation reste entre 0,984 et 0,986, celle du test entre 0,941 et 0,944.

**Conclusion.** Les conclusions des sections 5.1 à 5.4 ne dépendent ni de la graine ni du découpage. Le rôle dominant du TTL n'est pas un accident d'entraînement : il se retrouve à chaque fois, au même rang et avec le même poids. Deux limites : les hyperparamètres sont gardés fixes, et toutes les variantes viennent du même fichier d'entraînement. La stabilité mesurée ici est celle du modèle sur UNSW-NB15, pas sa capacité à se transposer à un autre réseau.

## 6. Analyse critique

### Le modèle apprend-il des comportements d'attaque ou des particularités du jeu ?

Notre réponse est qu'il apprend surtout une particularité du jeu : **l'identité des machines du banc d'essai**. Cette conclusion repose sur cinq observations indépendantes, chacune obtenue par une méthode différente.

1. **L'évaluation (section 4).** Le rappel reste élevé du fichier d'entraînement au fichier de test, mais le taux de fausses alertes triple (de 7 % à 24 % pour XGBoost). Les fausses alertes ne sont pas réparties au hasard : la quasi-totalité concerne des flux normaux dont le TTL source vaut 254 ou 62, les deux seules valeurs utilisées par les attaques. Les flux normaux avec un TTL de 31 ne sont jamais signalés.
2. **Les explications globales (section 5.1).** Trois mesures construites différemment placent `sttl` en tête et `ct_state_ttl` en deuxième position. Mélangées ensemble, les trois variables TTL font perdre à la forêt la moitié de son avance sur un modèle au hasard.
3. **Les explications locales (section 5.2).** Dans les cinq alertes expliquées, vraies comme fausses, `sttl` est la première raison de la décision. Une variable qui pousse de la même façon pour une attaque et pour un flux normal ne reconnaît pas l'attaque, elle reconnaît l'émetteur. Les erreurs opposent des flux presque identiques, que seules de petites différences de volume départagent.
4. **L'audit et l'ablation (sections 5.3 et 5.4).** Le TTL dépend du système d'exploitation de l'émetteur et du nombre de routeurs traversés, pas de la nature du trafic. Dans l'entraînement, aucune attaque n'a un TTL de 31, et toutes les familles d'attaques utilisent 62 ou 254. Retirer les variables TTL ne change presque rien : les scores des arbres restent identiques et les fausses alertes gardent exactement la même répartition par valeur de TTL. Un modèle entraîné sur le seul trafic normal reconnaît le groupe de machines sans aucune variable TTL, avec une ROC-AUC de 1,000.
5. **La stabilité (section 5.5).** Ces résultats se retrouvent à l'identique sur neuf entraînements avec d'autres graines et d'autres découpages.

Prises ensemble, ces observations montrent que l'artefact n'est pas une variable isolée qu'on pourrait retirer. C'est la disposition du banc d'essai : les attaques y sont produites par un ensemble précis de machines, et tout ce qui caractérise ces machines (TTL, délais réseau, débit) permet de les reconnaître. Un modèle qui s'appuie sur ces indices obtient d'excellents scores tant que le trafic normal ressemble à celui de l'entraînement, et produit des fausses alertes en masse dès qu'il change.

Cela ne signifie pas que les modèles n'apprennent **rien** du comportement des attaques. Certaines variables plausibles (volumes échangés, service, répétition des connexions) contribuent aussi, et la régression logistique, incapable de reconstruire l'identité des machines, garde une PR-AUC de 0,72 sans TTL, très au-dessus du hasard (0,36). Mais sur ce jeu de données, il n'est pas possible de séparer proprement la part du comportement de celle de l'artefact, puisque les deux sont liés par construction.

### Ce que cela dit de l'évaluation des détecteurs

**Un score élevé ne suffit pas.** Avec une évaluation correcte (doublons retirés, découpage sans fuite, métriques adaptées au déséquilibre), nos modèles atteignent une PR-AUC de 0,985 en validation. Rien dans ce chiffre ne laissait deviner le problème : il a fallu le fichier de test, fourni séparément par les auteurs et dont le trafic normal diffère de celui de l'entraînement, puis les explications pour le voir. Arp et al. (2022) décrivent ce piège sous le nom de corrélations parasites, et Geirhos et al. (2020) sous celui d'apprentissage par raccourci.

**Les explications ont joué un rôle de diagnostic.** L'explicabilité est souvent présentée comme un moyen de justifier les alertes auprès des analystes. Ici, elle a surtout servi à révéler que les alertes reposent sur un indice sans rapport avec les attaques. Une alerte « parce que TTL = 62 » n'aide pas un analyste à confirmer quoi que ce soit.

**La proportion d'attaques change tout.** Le jeu contient 36 % à 49 % d'attaques après nettoyage. Sur un vrai réseau, elles représentent une infime partie du trafic. À 1 % d'attaques, avec le taux de fausses alertes mesuré sur le test, moins de 4 % des alertes de XGBoost seraient de vraies attaques. C'est le problème du taux de base décrit par Axelsson (2000) : un détecteur d'intrusion doit avoir un taux de fausses alertes extrêmement faible pour être utilisable.

### Conséquences pratiques

Un détecteur de ce type ne devrait pas être déployé en l'état. Si nous devions en construire un pour un vrai réseau, nous retiendrions :

- entraîner et évaluer sur des données issues du réseau à protéger, ou au minimum sur plusieurs jeux de données, et tester sur un jeu différent de celui de l'entraînement ;
- exclure par principe les variables qui décrivent la machine ou le chemin réseau (TTL, délais de la poignée de main) plutôt que le comportement, en sachant que cela ne suffit pas toujours ;
- faire de l'analyse des explications une étape normale de la validation d'un modèle, au même titre que les métriques ;
- choisir le seuil avec l'équipe d'analystes, à partir d'un volume d'alertes qu'elle peut absorber.

## 7. Limites

- **Un seul jeu de données.** Toutes nos conclusions portent sur UNSW-NB15. Nous ne vérifions pas le comportement des modèles sur un autre jeu (par exemple CIC-IDS2017), ce qui aurait permis de mesurer directement la capacité de généralisation.
- **Le groupe de machines est approché par le TTL.** Les fichiers de partition ne contiennent pas les adresses IP. Nous ne pouvons pas nommer les machines, seulement les regrouper par valeur de TTL, ce qui reste un indicateur indirect.
- **Pas de découpage chronologique.** Les fichiers de partition n'ont pas de colonne de temps. Nous gardons la séparation des auteurs, sans pouvoir vérifier l'ordre temporel.
- **Effets du nettoyage.** Retirer les doublons et les flux du test déjà vus à l'entraînement rend l'évaluation plus honnête, mais nos scores ne sont plus directement comparables à ceux des articles qui utilisent les fichiers bruts.
- **Recherche d'hyperparamètres limitée.** Douze combinaisons tirées au hasard pour la forêt et XGBoost ; la meilleure profondeur de XGBoost est à la limite de l'espace exploré. Vu la proximité des deux modèles, cela ne change pas les conclusions.
- **Hyperparamètres gardés fixes** pour l'ablation et l'étude de stabilité, afin de n'isoler qu'un seul effet à la fois.
- **SHAP sur un échantillon** de 2 000 flux du test, comme l'impose le cahier des charges, et uniquement pour la forêt. Les explications de XGBoost n'ont pas été comparées (extension facultative).
- **Seuil choisi par le F1**, qui donne le même poids aux deux types d'erreurs. En production, le seuil dépendrait du coût réel des erreurs pour l'équipe.
- **Temps de calcul mesurés sur un ordinateur portable**, avec des écarts jusqu'à un facteur deux d'une exécution à l'autre. Seuls les ordres de grandeur sont interprétables.
- **Les variables de flux ne voient pas le contenu.** Plusieurs erreurs opposent des flux statistiquement presque identiques ; les distinguer demanderait d'examiner le contenu des paquets, hors du périmètre du projet.

## 8. Références

**Jeu de données**

- Moustafa, N. et Slay, J. (2015). UNSW-NB15: a comprehensive data set for network intrusion detection systems (UNSW-NB15 network data set). *Military Communications and Information Systems Conference (MilCIS)*, IEEE.
- Zoghi, Z. et Serpen, G. (2021). UNSW-NB15 Computer Security Dataset: Analysis through Visualization. arXiv:2101.05067.

**Modèles et outils**

- Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32.
- Chen, T. et Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *Proceedings of the 22nd ACM SIGKDD Conference on Knowledge Discovery and Data Mining*.
- Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.

**Explicabilité**

- Lundberg, S. M. et Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. *Advances in Neural Information Processing Systems 30 (NeurIPS)*.
- Lundberg, S. M. et al. (2020). From local explanations to global understanding with explainable AI for trees. *Nature Machine Intelligence*, 2, 56-67.
- Strobl, C., Boulesteix, A.-L., Zeileis, A. et Hothorn, T. (2007). Bias in random forest variable importance measures: Illustrations, sources and a solution. *BMC Bioinformatics*, 8, 25.

**Apprentissage automatique et sécurité**

- Arp, D. et al. (2022). Dos and Don'ts of Machine Learning in Computer Security. *31st USENIX Security Symposium*.
- Axelsson, S. (2000). The base-rate fallacy and the difficulty of intrusion detection. *ACM Transactions on Information and System Security*, 3(3), 186-205.
- Geirhos, R. et al. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence*, 2, 665-673.
- Sommer, R. et Paxson, V. (2010). Outside the Closed World: On Using Machine Learning for Network Intrusion Detection. *IEEE Symposium on Security and Privacy*.